# DIMER Notebook: Document Question Answering
## LayoutLM vs Pix2Struct DocVQA

**Notebook profile:** `TASK-INFERENCE`  
**Pedagogical mode:** `WORKSHOP`  
**DIMER Notebook Specification:** `2.1`  
**Comparison scope:** `MULTI-MODEL`  
**Standalone:** yes  
**Canonical workflow:** frozen comparison — no fine-tuning

This notebook compares two approaches to answering questions about document pages:

- **LayoutLM Document QA** — question + OCR words + 2-D word boxes → **extractive OCR span**
- **Pix2Struct DocVQA** — page pixels + question → **generated answer text**

Both models receive the **same held-out CORD-v2 receipt questions and gold answers**, but each sees the page through its native representation.

### Learning objectives

You will:

- distinguish OCR/layout-based extractive QA from OCR-free generative QA;
- measure ANLS, Exact Match, empty-answer rate and per-field behavior on one shared test set;
- inspect answer localization and answers not found in OCR;
- measure question-paraphrase sensitivity;
- probe unanswerable-question behavior;
- compare LayoutLM with and without spatial layout;
- compare Pix2Struct on clean and degraded page images; and
- interpret model quality alongside OCR dependency, traceability and compute cost.

> **Important:** the CORD gold answers in this notebook are deliberately constructed from contiguous OCR spans. This structurally aligns with LayoutLM's extractive answer mechanism. The notebook is a shared tutorial comparison, not a neutral benchmark of every Document-QA formulation.

## How to use this notebook

**Who it is for.** Learners who can run Python cells in Colab/Jupyter and are new to document QA; basic familiarity with OCR is helpful but not required.

**Runtime.** Use the documented GPU runtime for the LayoutLM and Pix2Struct comparison.

**How to run it.**
1. Select the documented runtime/accelerator.
2. Choose **Run all** for the canonical path; leave the default settings unchanged on your first pass.
3. Read the explanatory markdown while the notebook runs.
4. Sections marked **Infrastructure** support reproducibility, model acquisition, or orchestration. Run those cells as written; understanding their implementation is not a learning objective.

### Task at a glance

`document page + question → OCR/layout extractive QA or pixel-based generative QA → answer → exact/ANLS evaluation`

### Roadmap

1. Understand the task and its input/output contract.
2. Inspect and validate the built-in data or inputs.
3. Establish the baseline/reference behavior.
4. Run the model or multi-model comparison.
5. Inspect errors, disagreements, robustness, and/or resource tradeoffs.
6. Try one controlled change and explain what changed.
7. Write an evidence-based conclusion; optionally continue with BYOD.

### What successful execution looks like

You should finish with a validated input/sample, the notebook's principal baseline/reference, model outputs and evaluation results, at least one diagnostic or qualitative comparison, and machine-readable results/provenance where supported. Exact values can vary slightly across supported runtimes; focus on the defined metrics and the observed pattern.


## 1. Same task, different evidence

### LayoutLM

`question + OCR words + word boxes → layout-aware encoder → start/end span`

LayoutLM does **not** read page pixels. OCR is an external dependency.

### Pix2Struct

`page pixels + question rendered as a header → visual encoder → text decoder → generated answer`

Pix2Struct does not require external OCR, but its answer is not constrained to copy a page token.

The similar task label therefore hides different system boundaries, traceability, and error surfaces.

## 2. What each output means

LayoutLM can map its answer back to exact OCR word indices and page boxes. Its span score is an uncalibrated ranking signal, not probability that the answer is correct.

Pix2Struct returns generated text. The canonical API provides no answer confidence and no page localization. The notebook leaves those fields empty rather than fabricating symmetric outputs.

## 3. Configuration

The defaults define the canonical `Run all` path.

In [ ]:
USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ""  # @param {type:"string"}

MAX_NEW_TOKENS = 32  # @param {type:"integer"}
RUN_PARAPHRASE_EXPERIMENT = True  # @param {type:"boolean"}
PARAPHRASE_MAX_RECORDS = 20  # @param {type:"integer"}
RUN_UNANSWERABLE_PROBE = True  # @param {type:"boolean"}
UNANSWERABLE_PAGES = 5  # @param {type:"integer"}
RUN_MODALITY_ROBUSTNESS = True  # @param {type:"boolean"}
ROBUSTNESS_MAX_RECORDS = 20  # @param {type:"integer"}
OUTPUT_DIR = "outputs/document_qa_comparison"

if not 1 <= MAX_NEW_TOKENS <= 128:
    raise ValueError("MAX_NEW_TOKENS must be in 1..128")
if not 1 <= PARAPHRASE_MAX_RECORDS <= 20:
    raise ValueError("PARAPHRASE_MAX_RECORDS must be in 1..20")
if not 1 <= UNANSWERABLE_PAGES <= 10:
    raise ValueError("UNANSWERABLE_PAGES must be in 1..10")
print({
    "max_new_tokens":MAX_NEW_TOKENS,
    "paraphrase_experiment":RUN_PARAPHRASE_EXPERIMENT,
    "unanswerable_probe":RUN_UNANSWERABLE_PROBE,
    "modality_robustness":RUN_MODALITY_ROBUSTNESS,
    "use_byod":USE_BYOD,
})

## 4. Runtime

The two live carriers share the same Python 3.12 / PyTorch / Transformers runtime family. CORD parquet handling additionally uses PyArrow.

A Tesla T4 is recommended because the default path evaluates roughly 229 Pix2Struct questions with a 1.13 GB checkpoint.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
import importlib.metadata as importlib_metadata
import subprocess, sys

PINS = {
    "torch":"2.14.0","torchvision":"0.29.0","torchaudio":"2.11.0",
    "transformers":"4.57.6","safetensors":"0.8.0","numpy":"2.5.3",
    "pillow":"11.3.0","huggingface-hub":"0.36.2","pyarrow":"25.0.1",
}

def dist_version(name):
    try: return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError: return None

before = {k:dist_version(k) for k in PINS}
needed = [f"{k}=={v}" for k,v in PINS.items() if before[k] != v]
if needed:
    print("Installing pinned runtime:", needed)
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*needed])

after = {k:dist_version(k) for k in PINS}
bad = {k:(after[k],v) for k,v in PINS.items() if after[k] != v}
if bad:
    raise RuntimeError(f"Pinned install did not converge: {bad}")

stale=[]
for module_name,dist_name in [("torch","torch"),("transformers","transformers"),("numpy","numpy")]:
    module=sys.modules.get(module_name)
    if module is not None:
        runtime_v=getattr(module,"__version__",None)
        if runtime_v and not str(runtime_v).startswith(str(after[dist_name])):
            stale.append((module_name,runtime_v,after[dist_name]))
if stale:
    raise RuntimeError(
        "This kernel pre-imported packages replaced by the pinned install. "
        f"Start a fresh runtime and Run all again. Stale: {stale}"
    )

import numpy as np
import torch, torchvision, transformers, huggingface_hub, pyarrow
import pyarrow.parquet as pq
from PIL import Image, ImageDraw, ImageFont

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
RUNTIME = {
    "python":sys.version.split()[0],"torch":torch.__version__,"torchvision":torchvision.__version__,
    "transformers":transformers.__version__,"huggingface_hub":huggingface_hub.__version__,
    "pyarrow":pyarrow.__version__,"numpy":np.__version__,"pillow":importlib_metadata.version("pillow"),
    "device":DEVICE,"cuda_available":torch.cuda.is_available(),
}
if torch.cuda.is_available():
    RUNTIME["gpu_name"] = torch.cuda.get_device_name(0)
    RUNTIME["gpu_total_memory_bytes"] = torch.cuda.get_device_properties(0).total_memory
print(RUNTIME)

## 5. Immutable model provenance

The exact DIMER manifests for `impira/layoutlm-document-qa` and `google/pix2struct-docvqa-base` are embedded below.

Only manifest-listed files are fetched at immutable revisions. Every byte size and SHA-256 is checked before model loading. Upstream pickle alternatives are never used.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
import hashlib, json
from pathlib import Path
from huggingface_hub import hf_hub_download

LAYOUTLM_MANIFEST = json.loads(r"""{
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "layoutlm-document-qa",
  "modelId": "impira/layoutlm-document-qa",
  "revision": "beed3c4d02d86017ebca5bd0fdf210046b907aa6",
  "files": [
    {
      "path": "README.md",
      "bytes": 2326,
      "sha256": "40fd65fc734cc7eb73cc815465b8073bc4e5b7484406e537c7b8d763f88493f7"
    },
    {
      "path": "config.json",
      "bytes": 789,
      "sha256": "6d0fc068193109d0d053fa4de00963778beffbde05067c3e9f3454235044380f"
    },
    {
      "path": "merges.txt",
      "bytes": 456356,
      "sha256": "fe36cab26d4f4421ed725e10a2e9ddb7f799449c603a96e7f29b5a3c82a95862"
    },
    {
      "path": "model.safetensors",
      "bytes": 511200628,
      "sha256": "e4bbad3e4a1b5ae50c787b7afd6049a0bfa99fd823b50436e444e092ae2347b9"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 239,
      "sha256": "378eb3bf733eb16e65792d7e3fda5b8a4631387ca04d2015199c4d4f22ae554d"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1355881,
      "sha256": "33465117406b9007673e8ba283f7f1383d9b5094df947481af60eec94ed7d7bd"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 315,
      "sha256": "ea11996be5d083d63c72700810b18a6cfdf78c55131b754a75ae94e66b0ad6ab"
    },
    {
      "path": "vocab.json",
      "bytes": 798293,
      "sha256": "ed19656ea1707df69134c4af35c8ceda2cc9860bf2c3495026153a133670ab5e"
    }
  ],
  "totalBytes": 513814827
}""")
PIX2STRUCT_MANIFEST = json.loads(r"""{
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "pix2struct-docvqa-base",
  "modelId": "google/pix2struct-docvqa-base",
  "revision": "63f6b3de436e39f75c7a486881a9c2c14a7f4e89",
  "files": [
    {
      "path": "README.md",
      "bytes": 4476,
      "sha256": "794175546e80948e4efef30e95ebe859fdd26d07bcda26f738a1c97feb1e912a"
    },
    {
      "path": "config.json",
      "bytes": 4892,
      "sha256": "8d39973772a4218b555e30daecabdd5ea11aa1345dd711ff7f88fa90750b464f"
    },
    {
      "path": "model.safetensors",
      "bytes": 1129177976,
      "sha256": "067f7f314d87fa56daa5bcfaf36fa0b33ceebf7b7d4fae6a1e51ab7af64ee0b5"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 249,
      "sha256": "c84e4eebc84171d6069533d9f0147ec7b4afd02ab78697cb5c30f9419ef7dc45"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2201,
      "sha256": "5c87151ef0f72a99d1f766a4c418bd2a1f90aaa30a8e22fe5eca9641daebb64f"
    },
    {
      "path": "spiece.model",
      "bytes": 851388,
      "sha256": "7fd650335add59bed55a432186ca0437a09e185c2d241faab468a538fe6bcf94"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3265159,
      "sha256": "0af109b23840545ef2c286073f4373959badba1faa73c8557881d5126f6287c9"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2583,
      "sha256": "5fdb6767a49aca48fdfa43d0279321918185fc4997bdb3ea72bf3a6301a1b43d"
    }
  ],
  "totalBytes": 1133308924
}""")
LAYOUTLM_DIR = Path("weights/layoutlm-document-qa")
PIX2STRUCT_DIR = Path("weights/pix2struct-docvqa-base")
MANIFEST_NAME = "dimer-base-manifest.json"

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""): h.update(chunk)
    return h.hexdigest()

def stage_and_verify_model(root,manifest):
    root.mkdir(parents=True,exist_ok=True)
    (root/MANIFEST_NAME).write_text(json.dumps(manifest,indent=2),encoding="utf-8")
    for entry in manifest["files"]:
        path=root/entry["path"]
        if not path.is_file():
            hf_hub_download(
                repo_id=manifest["modelId"],filename=entry["path"],revision=manifest["revision"],
                local_dir=str(root),
            )
    for entry in manifest["files"]:
        path=root/entry["path"]
        if path.stat().st_size != entry["bytes"]:
            raise RuntimeError(f"{manifest['modelId']} {entry['path']} size mismatch")
        if sha256_file(path) != entry["sha256"]:
            raise RuntimeError(f"{manifest['modelId']} {entry['path']} SHA-256 mismatch")
    return {"model_id":manifest["modelId"],"revision":manifest["revision"],"files":len(manifest["files"]),"total_bytes":manifest["totalBytes"]}

layout_snapshot=stage_and_verify_model(LAYOUTLM_DIR,LAYOUTLM_MANIFEST)
pix_snapshot=stage_and_verify_model(PIX2STRUCT_DIR,PIX2STRUCT_MANIFEST)
print("LayoutLM:",layout_snapshot)
print("Pix2Struct:",pix_snapshot)

## 6. CORD-v2 provenance

The common evaluation set comes from `naver-clova-ix/cord-v2` at immutable revision:

`7f0115a4b758a71d6473b8d085751692da2fef98`

License: **CC BY 4.0**.

The notebook downloads and verifies the complete test and validation parquet shards so each selected receipt's **image and OCR/layout annotation are guaranteed to originate from the same pinned row**. The two files total about 476 MB.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
CORD_REPO="naver-clova-ix/cord-v2"
CORD_REVISION="7f0115a4b758a71d6473b8d085751692da2fef98"
CORD_LICENSE="CC BY 4.0"
CORD_FILES=json.loads(r"""{
  "test": {
    "path": "data/test-00000-of-00001-9c204eb3f4e11791.parquet",
    "bytes": 234202795,
    "sha256": "51c65f1788faff392abe2a0b55b023eb23e9be551c509138eaa3a832514224e7",
    "rows": 100
  },
  "validation": {
    "path": "data/validation-00000-of-00001-cc3c5779fe22e8ca.parquet",
    "bytes": 242080800,
    "sha256": "0d0f6dac11fdcc549de2746aa9f53136a3bc22a2a1aff2b0b847f7622ad60c15",
    "rows": 100
  }
}""")
CORD_DIR=Path("weights/cord-v2-full")
CORD_DIR.mkdir(parents=True,exist_ok=True)

cord_paths={}
for split,spec in CORD_FILES.items():
    local=Path(hf_hub_download(
        repo_id=CORD_REPO,repo_type="dataset",filename=spec["path"],revision=CORD_REVISION,
        local_dir=str(CORD_DIR),
    ))
    if local.stat().st_size != spec["bytes"]:
        raise RuntimeError(f"{split} shard size mismatch")
    digest=sha256_file(local)
    if digest != spec["sha256"]:
        raise RuntimeError(f"{split} shard SHA-256 mismatch")
    cord_paths[split]=local
    print(split, local.stat().st_size, digest)

## 7. Reconstruct page pixels, OCR words and layout

Each parquet row contains the receipt image and a `ground_truth` JSON annotation. The parser reproduces the current LayoutLM DIMER semantics:

- each OCR quadrilateral becomes an axis-aligned pixel box;
- blank or degenerate words are dropped;
- each semantic line tracks the contiguous span of its **value** words;
- image dimensions must agree with annotation dimensions.

In [ ]:
import io, random, re
from collections import defaultdict

QUESTION_TEMPLATES={
    "total.total_price":"What is the total amount?",
    "sub_total.subtotal_price":"What is the subtotal?",
    "sub_total.tax_price":"What is the tax amount?",
    "sub_total.service_price":"What is the service charge?",
    "sub_total.discount_price":"What is the discount amount?",
    "total.cashprice":"How much cash was paid?",
    "total.changeprice":"How much change was given?",
    "total.creditcardprice":"How much was paid by card?",
    "total.menuqty_cnt":"How many items were bought?",
    "menu.nm":"What is the name of the first item?",
}
FIRST_ITEM_CATEGORY="menu.nm"
SAMPLE_SEED=42
SAMPLE_PAGE_SPLIT={"train":119,"validation":30,"test":50}

def quad_to_box(quad,width,height):
    xs=[float(quad[k]) for k in ("x1","x2","x3","x4")]
    ys=[float(quad[k]) for k in ("y1","y2","y3","y4")]
    x0,y0=max(0.0,min(xs)),max(0.0,min(ys))
    x1,y1=min(float(width),max(xs)),min(float(height),max(ys))
    return None if x1<=x0 or y1<=y0 else [x0,y0,x1,y1]

def decode_image_cell(cell):
    if isinstance(cell,Image.Image): return cell.convert("RGB")
    if isinstance(cell,dict):
        data=cell.get("bytes")
        if data is not None:
            im=Image.open(io.BytesIO(data)); im.load(); return im.convert("RGB")
        path=cell.get("path")
        if path and Path(path).is_file():
            im=Image.open(path); im.load(); return im.convert("RGB")
    if isinstance(cell,(bytes,bytearray)):
        im=Image.open(io.BytesIO(cell)); im.load(); return im.convert("RGB")
    raise TypeError(f"Unsupported CORD image representation: {type(cell).__name__}")

def page_from_ground_truth(ground_truth,page_id,image,source_split,source_row):
    data=json.loads(ground_truth) if isinstance(ground_truth,str) else ground_truth
    size=data["meta"]["image_size"]
    width,height=int(size["width"]),int(size["height"])
    if image.size != (width,height):
        raise RuntimeError(f"{page_id}: image {image.size} != annotation {(width,height)}")
    words,boxes,lines=[],[],[]
    for line in data.get("valid_line",[]):
        start=len(words); values=[]
        for word in line.get("words",[]):
            text=" ".join(str(word.get("text","")).split())
            box=quad_to_box(word["quad"],width,height) if text else None
            if box is None: continue
            if not int(word.get("is_key",0)): values.append(len(words))
            words.append(text); boxes.append(box)
        if len(words)>start:
            contiguous=bool(values) and values==list(range(values[0],values[-1]+1))
            lines.append({
                "category":str(line.get("category","")),"group_id":int(line.get("group_id",-1)),
                "start":start,"end":len(words)-1,
                "value_start":values[0] if contiguous else None,
                "value_end":values[-1] if contiguous else None,
            })
    pixel_digest=hashlib.sha256(f"{image.width}x{image.height}:".encode()+image.tobytes()).hexdigest()
    ocr_digest=hashlib.sha256(json.dumps([words,boxes],separators=(",",":"),ensure_ascii=False).encode()).hexdigest()
    return {
        "id":page_id,"image":image,"words":words,"boxes":boxes,"image_size":[width,height],"lines":lines,
        "source_split":source_split,"source_row":source_row,"pixel_sha256":pixel_digest,"ocr_sha256":ocr_digest,
    }

pages_by_source={}
for split,path in cord_paths.items():
    table=pq.read_table(path,columns=["image","ground_truth"])
    rows=table.to_pylist()
    if len(rows)!=CORD_FILES[split]["rows"]:
        raise RuntimeError(f"{split}: expected {CORD_FILES[split]['rows']} rows, got {len(rows)}")
    pages=[]
    for idx,row in enumerate(rows):
        image=decode_image_cell(row["image"])
        pages.append(page_from_ground_truth(row["ground_truth"],f"{split}-{idx:03d}",image,split,idx))
    pages_by_source[split]=pages
print({k:len(v) for k,v in pages_by_source.items()})

## 8. Reproduce the existing 119 / 30 / 50 page split

A field becomes a QA record only when its value is a valid contiguous OCR span. Pages are deduplicated by lower-cased OCR word sequence **before** seeded splitting.

The existing DIMER contract should reproduce:

- 199 supported unique pages;
- 119 / 30 / 50 train/validation/test pages;
- 595 / 152 / 229 QA records.

The notebook evaluates only the held-out 50-page / 229-question test set.

In [ ]:
def questions_for_page(page):
    by_category=defaultdict(list)
    for line in page["lines"]: by_category[line["category"]].append(line)
    out=[]
    for category,question in QUESTION_TEMPLATES.items():
        candidates=by_category.get(category,[])
        if not candidates: continue
        if category==FIRST_ITEM_CATEGORY:
            topmost=min(candidates,key=lambda line:(page["boxes"][line["start"]][1],line["start"]))
            same_group=[line for line in candidates if line["group_id"]==topmost["group_id"]]
            if len(same_group)!=1: continue
            line=topmost
        elif len(candidates)==1:
            line=candidates[0]
        else:
            continue
        if line["value_start"] is None: continue
        start,end=int(line["value_start"]),int(line["value_end"])
        answer=" ".join(page["words"][start:end+1])
        out.append({
            "page_id":page["id"],"field":category,"question":question,"words":page["words"],"boxes":page["boxes"],
            "image_size":page["image_size"],"answer_start":start,"answer_end":end,"answers":[answer],
        })
    return out

pool=[dict(page) for split in sorted(pages_by_source) for page in pages_by_source[split]]
seen=set(); supported=[]
for page in pool:
    key=tuple(str(w).lower() for w in page["words"])
    if key in seen or not questions_for_page(page): continue
    seen.add(key); supported.append(page)

random.Random(SAMPLE_SEED).shuffle(supported)
page_splits={}; qa_splits={}; offset=0
for name in ("train","validation","test"):
    chosen=supported[offset:offset+SAMPLE_PAGE_SPLIT[name]]; offset+=SAMPLE_PAGE_SPLIT[name]
    page_splits[name]=chosen
    rows=[]
    for page in chosen: rows.extend(questions_for_page(page))
    qa_splits[name]=[{"id":f"{name}-{i:04d}",**row} for i,row in enumerate(rows)]

page_counts={k:len(v) for k,v in page_splits.items()}
qa_counts={k:len(v) for k,v in qa_splits.items()}
if len(supported)!=199: raise RuntimeError(f"Expected 199 supported unique pages, got {len(supported)}")
if page_counts!={"train":119,"validation":30,"test":50}: raise RuntimeError(page_counts)
if qa_counts!={"train":595,"validation":152,"test":229}: raise RuntimeError(qa_counts)

owners={}
for split,pages in page_splits.items():
    for page in pages:
        if page["id"] in owners: raise RuntimeError(f"Page leakage: {page['id']}")
        owners[page["id"]]=split

test_records=qa_splits["test"]
test_pages={p["id"]:p for p in page_splits["test"]}
qa_digest=hashlib.sha256("\n".join(
    json.dumps([r["id"],r["page_id"],r["field"],r["question"],r["answers"],r["answer_start"],r["answer_end"]],separators=(",",":"),ensure_ascii=False)
    for r in test_records
).encode()).hexdigest()
print({"supported_unique_pages":len(supported),"page_splits":page_counts,"qa_splits":qa_counts,"test_qa_digest":qa_digest})

## 9. Common answer metrics

Both models and both OCR baselines use exactly the same:

- **ANLS** with the standard 0.5 threshold;
- **Exact Match** after the same normalization;
- **empty-answer rate**.

The notebook also asks whether a prediction exactly matches a contiguous normalized OCR span. `answer_in_ocr` is diagnostic, not a correctness metric.

In [ ]:
PUNCT_RE=re.compile(r"[^\w\s]")
ANLS_THRESHOLD=0.5

def normalize_answer(text): return " ".join(PUNCT_RE.sub(" ",str(text).lower()).split())

def levenshtein(a,b):
    prev=list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        cur=[i]
        for j,cb in enumerate(b,1): cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(ca!=cb)))
        prev=cur
    return prev[-1]

def anls(prediction,golds,threshold=ANLS_THRESHOLD):
    pred=normalize_answer(prediction); best=0.0
    for gold in golds:
        ref=normalize_answer(gold); longest=max(len(pred),len(ref))
        sim=1.0 if longest==0 else 1.0-levenshtein(pred,ref)/longest
        best=max(best,sim)
    return best if best>=threshold else 0.0

def exact_match(prediction,golds):
    pred=normalize_answer(prediction)
    return any(pred==normalize_answer(g) for g in golds)

def answer_in_ocr(prediction,words,max_span_words=20):
    target=normalize_answer(prediction)
    if not target: return False
    cleaned=[normalize_answer(w) for w in words]
    for start in range(len(cleaned)):
        parts=[]
        for end in range(start,min(len(cleaned),start+max_span_words)):
            if cleaned[end]: parts.append(cleaned[end])
            joined=" ".join(parts)
            if joined==target: return True
            if len(joined)>len(target)+32: break
    return False

def corpus_metrics(rows):
    if not rows: raise ValueError("No prediction rows")
    return {
        "n":len(rows),"anls":float(np.mean([r["anls"] for r in rows])),
        "exact_match":float(np.mean([r["exact_match"] for r in rows])),
        "empty_rate":float(np.mean([r["empty"] for r in rows])),
        "answer_in_ocr_rate":float(np.mean([r["answer_in_ocr"] for r in rows])),
    }

## 10. OCR baselines

Two simple references reuse the current LayoutLM carrier's baseline semantics:

- **last-number** — the last OCR word containing a digit;
- **keyword lookup** — find the page word most similar to a content word in the question, then return the next numeric OCR word.

Because both already assume OCR is available, they share more of LayoutLM's information boundary than Pix2Struct's.

> **Before you run it:** predict whether this simple reference will be easy or difficult for the learned model(s) to beat. Record the baseline before interpreting the more complex result.

In [ ]:
DIGIT_RE=re.compile(r"\d")
QUESTION_STOP_WORDS=frozenset([
    "what","is","the","how","much","many","was","were","paid","given","by","of","name",
    "first","item","items","bought","amount","charge"
])

def last_number_answer(words):
    for word in reversed(words):
        if DIGIT_RE.search(word): return word
    return ""

def prefix_overlap(a,b):
    n=0
    for x,y in zip(a,b):
        if x!=y: break
        n+=1
    return n

def keyword_lookup_answer(question,words,min_overlap=3):
    content=[w for w in normalize_answer(question).split() if w not in QUESTION_STOP_WORDS]
    page=[normalize_answer(w) for w in words]
    best_index,best_overlap=None,min_overlap-1
    for idx,word in enumerate(page):
        overlap=max((prefix_overlap(word,q) for q in content),default=0)
        if overlap>=best_overlap and overlap>=min_overlap:
            best_index,best_overlap=idx,overlap
    if best_index is not None:
        for word in words[best_index+1:]:
            if DIGIT_RE.search(word): return word
    return last_number_answer(words)

def baseline_rows(name,answer_fn):
    rows=[]
    for r in test_records:
        answer=answer_fn(r)
        rows.append({
            "record_id":r["id"],"page_id":r["page_id"],"field":r["field"],"question":r["question"],"gold_answer":r["answers"][0],
            "model":name,"answer":answer,"normalized_answer":normalize_answer(answer),"anls":anls(answer,r["answers"]),
            "exact_match":exact_match(answer,r["answers"]),"empty":not bool(answer.strip()),
            "answer_in_ocr":answer_in_ocr(answer,r["words"]),"latency_seconds":None,
        })
    return rows

last_number_rows=baseline_rows("Last-number baseline",lambda r:last_number_answer(r["words"]))
keyword_rows=baseline_rows("Keyword lookup",lambda r:keyword_lookup_answer(r["question"],r["words"]))
print("Last-number:",corpus_metrics(last_number_rows))
print("Keyword lookup:",corpus_metrics(keyword_rows))

## 11. LayoutLM — extractive QA over OCR + layout

LayoutLM receives the question, OCR words and pixel boxes normalized to a 0–1000 grid. It never sees the page image.

Long inputs are split into overlapping 512-token windows. Only document tokens may start/end an answer, and answer spans are capped at 15 tokens.

In [ ]:
import gc, time
from transformers import AutoTokenizer, LayoutLMForQuestionAnswering

LAYOUTLM_ID=LAYOUTLM_MANIFEST["modelId"]; LAYOUTLM_REVISION=LAYOUTLM_MANIFEST["revision"]
MAX_SEQ_LEN=512; DOC_STRIDE=128; MAX_ANSWER_TOKENS=15; BOX_GRID=1000

def normalize_box(box,width,height):
    x0,y0,x1,y1=[float(v) for v in box]
    return [int(BOX_GRID*x0/width),int(BOX_GRID*y0/height),int(BOX_GRID*x1/width),int(BOX_GRID*y1/height)]

layout_tokenizer=AutoTokenizer.from_pretrained(str(LAYOUTLM_DIR),local_files_only=True,trust_remote_code=False)
if not layout_tokenizer.is_fast: raise RuntimeError("LayoutLM requires a fast tokenizer")
t0=time.perf_counter()
layout_model=LayoutLMForQuestionAnswering.from_pretrained(
    str(LAYOUTLM_DIR),local_files_only=True,trust_remote_code=False,dtype=torch.float32
).to(DEVICE).eval()
layout_load_seconds=time.perf_counter()-t0
for p in layout_model.parameters(): p.requires_grad_(False)
layout_parameter_count=sum(p.numel() for p in layout_model.parameters())
if layout_parameter_count!=127_792_898: raise RuntimeError(f"Unexpected LayoutLM parameter count: {layout_parameter_count}")
layout_sep_id=layout_tokenizer.sep_token_id

def layout_encode(question,words):
    return layout_tokenizer(
        text=question.split(),text_pair=words,is_split_into_words=True,max_length=MAX_SEQ_LEN,stride=DOC_STRIDE,
        truncation="only_second",return_overflowing_tokens=True,return_token_type_ids=True,
        padding="max_length",return_tensors="pt",
    )

def layout_window_boxes(encoding,window,grid_boxes):
    out=[]
    for input_id,sequence_id,word_id in zip(
        encoding["input_ids"][window].tolist(),encoding.sequence_ids(window),encoding.word_ids(window),strict=True
    ):
        if sequence_id==1: out.append(list(grid_boxes[word_id]))
        elif input_id==layout_sep_id: out.append([BOX_GRID]*4)
        else: out.append([0]*4)
    return out

def layout_answer(question,words,boxes,image_size):
    width,height=image_size
    if not words or len(words)!=len(boxes): raise ValueError("LayoutLM requires one box per OCR word")
    q=" ".join(str(question).split())
    if not q or len(q)>256: raise ValueError("Question must contain 1..256 characters")
    for box in boxes:
        x0,y0,x1,y1=[float(v) for v in box]
        if not (0<=x0<=x1<=width and 0<=y0<=y1<=height): raise ValueError("OCR box outside page")
    grid_boxes=[normalize_box(b,width,height) for b in boxes]
    enc=layout_encode(q,words); n_windows=int(enc["input_ids"].shape[0])
    best={"start":None,"end":None,"score":0.0,"n_windows":n_windows}
    model_device=next(layout_model.parameters()).device
    for window in range(n_windows):
        sequence_ids=enc.sequence_ids(window); word_ids=enc.word_ids(window)
        inputs={
            "input_ids":enc["input_ids"][window].unsqueeze(0).to(model_device),
            "attention_mask":enc["attention_mask"][window].unsqueeze(0).to(model_device),
            "token_type_ids":enc["token_type_ids"][window].unsqueeze(0).to(model_device),
            "bbox":torch.tensor(layout_window_boxes(enc,window,grid_boxes),dtype=torch.long).unsqueeze(0).to(model_device),
        }
        with torch.inference_mode(): outputs=layout_model(**inputs)
        allowed=torch.tensor([sid==1 for sid in sequence_ids],device=model_device)
        start=outputs.start_logits[0].float().masked_fill(~allowed,float("-inf")).softmax(-1)
        end=outputs.end_logits[0].float().masked_fill(~allowed,float("-inf")).softmax(-1)
        candidates=start[:,None]*end[None,:]
        candidates=torch.triu(candidates)-torch.triu(candidates,diagonal=MAX_ANSWER_TOKENS)
        flat=int(candidates.argmax()); s_idx,e_idx=divmod(flat,candidates.shape[1]); score=float(candidates[s_idx,e_idx])
        if score>best["score"] and word_ids[s_idx] is not None and word_ids[e_idx] is not None:
            best={"start":int(word_ids[s_idx]),"end":int(word_ids[e_idx]),"score":score,"n_windows":n_windows}
    if best["start"] is None:
        answer=""; union_box=None
    else:
        answer=" ".join(words[best["start"]:best["end"]+1]); selected=boxes[best["start"]:best["end"]+1]
        union_box=[min(b[0] for b in selected),min(b[1] for b in selected),max(b[2] for b in selected),max(b[3] for b in selected)]
    return {**best,"answer":answer,"answer_union_box":union_box}

## 12. Held-out LayoutLM evaluation

One warm-up call is excluded from latency. CUDA synchronization surrounds each measured question when a GPU is used.

Model download, model loading and OCR production are not included in per-question latency.

> **What to notice.** Compare the learned-model result with the baseline/reference first, then use the secondary diagnostics to explain the behavior. Do not infer a universal model ranking from one tutorial sample and configuration.

In [ ]:
warm=test_records[0]
_=layout_answer(warm["question"],warm["words"],warm["boxes"],warm["image_size"])
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
layout_rows=[]; layout_times=[]
for i,r in enumerate(test_records):
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0=time.perf_counter(); result=layout_answer(r["question"],r["words"],r["boxes"],r["image_size"])
    if torch.cuda.is_available(): torch.cuda.synchronize()
    dt=time.perf_counter()-t0; layout_times.append(dt); answer=result["answer"]
    layout_rows.append({
        "record_id":r["id"],"page_id":r["page_id"],"field":r["field"],"question":r["question"],"gold_answer":r["answers"][0],
        "model":"LayoutLM","answer":answer,"normalized_answer":normalize_answer(answer),"anls":anls(answer,r["answers"]),
        "exact_match":exact_match(answer,r["answers"]),"empty":not bool(answer.strip()),"answer_in_ocr":answer_in_ocr(answer,r["words"]),
        "latency_seconds":dt,"start_word":result["start"],"end_word":result["end"],"span_score":result["score"],
        "n_windows":result["n_windows"],"answer_union_box":result["answer_union_box"],"new_tokens":None,"truncated":None,
    })
    if (i+1)%50==0: print(f"LayoutLM: {i+1}/{len(test_records)}")
layout_peak_gpu=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
layout_metrics=corpus_metrics(layout_rows)
print(layout_metrics)

## 13. LayoutLM modality experiment — remove spatial layout

The same OCR text is retained, but every document word receives the same zero box. This removes useful 2-D layout while preserving question and token content.

The experiment uses a predetermined subset and does not alter the main evaluation.

In [ ]:
experiment_records=sorted(test_records,key=lambda r:r["id"])[:ROBUSTNESS_MAX_RECORDS]
layout_no_layout_rows=[]
if RUN_MODALITY_ROBUSTNESS:
    for r in experiment_records:
        zero_boxes=[[0.0,0.0,0.0,0.0] for _ in r["boxes"]]
        result=layout_answer(r["question"],r["words"],zero_boxes,r["image_size"])
        canonical=next(x for x in layout_rows if x["record_id"]==r["id"])
        layout_no_layout_rows.append({
            "model":"LayoutLM","record_id":r["id"],"experiment":"no_layout",
            "canonical_answer":canonical["answer"],"variant_answer":result["answer"],
            "canonical_anls":canonical["anls"],"variant_anls":anls(result["answer"],r["answers"]),
            "canonical_exact":canonical["exact_match"],"variant_exact":exact_match(result["answer"],r["answers"]),
        })
    for row in layout_no_layout_rows: row["delta_anls"]=row["variant_anls"]-row["canonical_anls"]
    print({"canonical_mean_anls":float(np.mean([r["canonical_anls"] for r in layout_no_layout_rows])),
           "no_layout_mean_anls":float(np.mean([r["variant_anls"] for r in layout_no_layout_rows]))})
else:
    print("Layout robustness experiment disabled.")

## 14. Question paraphrases and unanswerable questions

Paraphrase records are selected by stable record ID before results are inspected.

The unanswerable probe asks for a **loyalty membership number**, which is not one of the CORD fields used to generate the evaluation set. It has no fabricated gold answer and is therefore behavior evidence only.

In [ ]:
PARAPHRASES={
    "total.total_price":"How much is the total?","sub_total.subtotal_price":"How much is the subtotal?",
    "sub_total.tax_price":"What tax amount is shown?","sub_total.service_price":"How much is the service charge?",
    "sub_total.discount_price":"What discount was applied?","total.cashprice":"How much cash did the customer pay?",
    "total.changeprice":"How much change was returned?","total.creditcardprice":"How much was charged to the card?",
    "total.menuqty_cnt":"How many items are on the receipt?","menu.nm":"What is the first item listed?",
}
paraphrase_records=sorted(test_records,key=lambda r:r["id"])[:PARAPHRASE_MAX_RECORDS]
layout_paraphrase_rows=[]
if RUN_PARAPHRASE_EXPERIMENT:
    for r in paraphrase_records:
        variant_q=PARAPHRASES[r["field"]]; result=layout_answer(variant_q,r["words"],r["boxes"],r["image_size"])
        canonical=next(x for x in layout_rows if x["record_id"]==r["id"])
        layout_paraphrase_rows.append({
            "model":"LayoutLM","record_id":r["id"],"field":r["field"],"canonical_question":r["question"],
            "paraphrased_question":variant_q,"canonical_answer":canonical["answer"],"paraphrased_answer":result["answer"],
            "canonical_anls":canonical["anls"],"paraphrased_anls":anls(result["answer"],r["answers"]),
            "answer_stable":normalize_answer(canonical["answer"])==normalize_answer(result["answer"]),
        })

probe_pages=sorted(test_pages)[:UNANSWERABLE_PAGES]
UNANSWERABLE_QUESTION="What is the loyalty membership number?"
layout_unanswerable_rows=[]
if RUN_UNANSWERABLE_PROBE:
    for page_id in probe_pages:
        page=test_pages[page_id]; result=layout_answer(UNANSWERABLE_QUESTION,page["words"],page["boxes"],page["image_size"])
        layout_unanswerable_rows.append({
            "model":"LayoutLM","page_id":page_id,"question":UNANSWERABLE_QUESTION,"answer":result["answer"],
            "empty":not bool(result["answer"].strip()),"answer_in_ocr":answer_in_ocr(result["answer"],page["words"]),
            "span_score":result["score"],"new_tokens":None,"truncated":None,
        })
print("Paraphrase rows:",len(layout_paraphrase_rows))
print("Unanswerable rows:",len(layout_unanswerable_rows))

## 15. Release LayoutLM before loading Pix2Struct

Only normalized predictions and diagnostics remain in memory. This keeps the notebook's peak accelerator requirement bounded.

In [ ]:
del layout_model,layout_tokenizer
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("LayoutLM released.")

## 16. Pix2Struct — pixels + question, generated answer

Pix2Struct's VQA processor renders the question above the page and scales the composite into at most 2,048 16×16 patches.

The notebook supplies Pillow's bundled Aileron font bytes directly so the processor never performs the upstream unpinned Arial-font download.

In [ ]:
from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor
PIX_ID=PIX2STRUCT_MANIFEST["modelId"]; PIX_REVISION=PIX2STRUCT_MANIFEST["revision"]
pix_processor=Pix2StructProcessor.from_pretrained(str(PIX2STRUCT_DIR),local_files_only=True,trust_remote_code=False)
if not getattr(pix_processor.image_processor,"is_vqa",False): raise RuntimeError("Pix2Struct processor is not VQA mode")
font_obj=ImageFont.load_default(size=36); FONT_BYTES=getattr(font_obj,"font_bytes",None)
if not FONT_BYTES: raise RuntimeError("Pillow bundled TrueType font unavailable")
FONT_BYTES=bytes(FONT_BYTES); FONT_SHA256=hashlib.sha256(FONT_BYTES).hexdigest()
t0=time.perf_counter()
pix_model=Pix2StructForConditionalGeneration.from_pretrained(
    str(PIX2STRUCT_DIR),local_files_only=True,trust_remote_code=False,dtype=torch.float32
).to(DEVICE).eval()
pix_load_seconds=time.perf_counter()-t0
for p in pix_model.parameters(): p.requires_grad_(False)
pix_parameter_count=sum(p.numel() for p in pix_model.parameters())

def pix_answer(image,question,max_new_tokens=MAX_NEW_TOKENS):
    q=" ".join(str(question).split())
    if not q or len(q)>256: raise ValueError("Question must contain 1..256 characters")
    if min(image.size)<16 or max(image.size)>4096: raise ValueError("Pix2Struct image side outside 16..4096")
    if isinstance(max_new_tokens,bool) or not isinstance(max_new_tokens,int) or not 1<=max_new_tokens<=128:
        raise ValueError("max_new_tokens must be an int in 1..128")
    inputs=pix_processor.image_processor(image.convert("RGB"),header_text=q,return_tensors="pt",font_bytes=FONT_BYTES).to(DEVICE)
    with torch.inference_mode():
        generated=pix_model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False)
    answer=pix_processor.tokenizer.batch_decode(generated,skip_special_tokens=True)[0].strip()
    new_tokens=int(generated[0].shape[0])-1
    return {"answer":answer,"new_tokens":new_tokens,"truncated":new_tokens>=max_new_tokens}

print({"parameters":pix_parameter_count,"load_seconds":pix_load_seconds,"font_sha256":FONT_SHA256})

## 17. Held-out Pix2Struct evaluation

Each CORD question is asked against the **same receipt image** whose OCR/layout representation LayoutLM received.

Pix2Struct has no canonical answer-confidence output, so the comparison keeps that field null.

In [ ]:
warm=test_records[0]
_=pix_answer(test_pages[warm["page_id"]]["image"],warm["question"])
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
pix_rows=[]; pix_times=[]
for i,r in enumerate(test_records):
    page=test_pages[r["page_id"]]
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0=time.perf_counter(); result=pix_answer(page["image"],r["question"])
    if torch.cuda.is_available(): torch.cuda.synchronize()
    dt=time.perf_counter()-t0; pix_times.append(dt); answer=result["answer"]
    pix_rows.append({
        "record_id":r["id"],"page_id":r["page_id"],"field":r["field"],"question":r["question"],"gold_answer":r["answers"][0],
        "model":"Pix2Struct","answer":answer,"normalized_answer":normalize_answer(answer),"anls":anls(answer,r["answers"]),
        "exact_match":exact_match(answer,r["answers"]),"empty":not bool(answer.strip()),"answer_in_ocr":answer_in_ocr(answer,r["words"]),
        "latency_seconds":dt,"start_word":None,"end_word":None,"span_score":None,"n_windows":None,"answer_union_box":None,
        "new_tokens":result["new_tokens"],"truncated":result["truncated"],
    })
    if (i+1)%25==0: print(f"Pix2Struct: {i+1}/{len(test_records)}")
pix_peak_gpu=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
pix_metrics=corpus_metrics(pix_rows)
print(pix_metrics)

## 18. Pix2Struct modality experiment — reduce page detail

The same predetermined QA subset is rerun after each page is downsampled to half its original linear resolution and then resized back.

The question and gold answer do not change. This is an exploratory image-quality probe, not a recommended preprocessing policy.

In [ ]:
pix_degraded_rows=[]
if RUN_MODALITY_ROBUSTNESS:
    for r in experiment_records:
        page=test_pages[r["page_id"]]; image=page["image"]
        low=image.resize((max(16,image.width//2),max(16,image.height//2)),Image.Resampling.BILINEAR)
        degraded=low.resize(image.size,Image.Resampling.BILINEAR)
        result=pix_answer(degraded,r["question"]); canonical=next(x for x in pix_rows if x["record_id"]==r["id"])
        pix_degraded_rows.append({
            "model":"Pix2Struct","record_id":r["id"],"experiment":"low_resolution",
            "canonical_answer":canonical["answer"],"variant_answer":result["answer"],
            "canonical_anls":canonical["anls"],"variant_anls":anls(result["answer"],r["answers"]),
            "canonical_exact":canonical["exact_match"],"variant_exact":exact_match(result["answer"],r["answers"]),
        })
    for row in pix_degraded_rows: row["delta_anls"]=row["variant_anls"]-row["canonical_anls"]
    print({"canonical_mean_anls":float(np.mean([r["canonical_anls"] for r in pix_degraded_rows])),
           "degraded_mean_anls":float(np.mean([r["variant_anls"] for r in pix_degraded_rows]))})
else:
    print("Pix2Struct robustness experiment disabled.")

## 19. Pix2Struct paraphrase and unanswerable probes

The exact same record and page subsets used for LayoutLM are reused here.

In [ ]:
pix_paraphrase_rows=[]
if RUN_PARAPHRASE_EXPERIMENT:
    for r in paraphrase_records:
        variant_q=PARAPHRASES[r["field"]]; result=pix_answer(test_pages[r["page_id"]]["image"],variant_q)
        canonical=next(x for x in pix_rows if x["record_id"]==r["id"])
        pix_paraphrase_rows.append({
            "model":"Pix2Struct","record_id":r["id"],"field":r["field"],"canonical_question":r["question"],
            "paraphrased_question":variant_q,"canonical_answer":canonical["answer"],"paraphrased_answer":result["answer"],
            "canonical_anls":canonical["anls"],"paraphrased_anls":anls(result["answer"],r["answers"]),
            "answer_stable":normalize_answer(canonical["answer"])==normalize_answer(result["answer"]),
        })

pix_unanswerable_rows=[]
if RUN_UNANSWERABLE_PROBE:
    for page_id in probe_pages:
        page=test_pages[page_id]; result=pix_answer(page["image"],UNANSWERABLE_QUESTION)
        pix_unanswerable_rows.append({
            "model":"Pix2Struct","page_id":page_id,"question":UNANSWERABLE_QUESTION,"answer":result["answer"],
            "empty":not bool(result["answer"].strip()),"answer_in_ocr":answer_in_ocr(result["answer"],page["words"]),
            "span_score":None,"new_tokens":result["new_tokens"],"truncated":result["truncated"],
        })
print("Paraphrase rows:",len(pix_paraphrase_rows))
print("Unanswerable rows:",len(pix_unanswerable_rows))

## 20. Main comparison

ANLS and Exact Match are computed identically for all systems.

Interpret the table with the information boundaries in mind: LayoutLM sees OCR + layout, Pix2Struct sees pixels, and the gold answers are OCR spans.

In [ ]:
systems=[("Last-number baseline",last_number_rows),("Keyword lookup",keyword_rows),("LayoutLM",layout_rows),("Pix2Struct",pix_rows)]
system_metrics={name:corpus_metrics(rows) for name,rows in systems}
print(f"{'System':<24} {'ANLS':>8} {'Exact':>8} {'Empty':>8} {'In OCR':>8}")
print("-"*62)
for name,_ in systems:
    m=system_metrics[name]
    print(f"{name:<24} {m['anls']:>8.3f} {m['exact_match']:>8.3f} {m['empty_rate']:>8.3f} {m['answer_in_ocr_rate']:>8.3f}")

## 21. Per-field behavior

Aggregate metrics can hide field-specific errors. Report support, ANLS, Exact Match, empty-answer rate and OCR containment for every question field.

In [ ]:
def field_metrics(rows):
    by_field=defaultdict(list)
    for row in rows: by_field[row["field"]].append(row)
    result=[]
    for field in QUESTION_TEMPLATES:
        group=by_field.get(field,[])
        if not group: continue
        result.append({
            "field":field,"support":len(group),"mean_anls":float(np.mean([r["anls"] for r in group])),
            "exact_match":float(np.mean([r["exact_match"] for r in group])),"empty_rate":float(np.mean([r["empty"] for r in group])),
            "answer_in_ocr_rate":float(np.mean([r["answer_in_ocr"] for r in group])),
        })
    return result
layout_field_metrics=field_metrics(layout_rows); pix_field_metrics=field_metrics(pix_rows)
layout_by_field={r["field"]:r for r in layout_field_metrics}; pix_by_field={r["field"]:r for r in pix_field_metrics}
print(f"{'Field':<30} {'N':>4} {'L-ANLS':>8} {'P-ANLS':>8} {'L-EM':>7} {'P-EM':>7}")
for field in QUESTION_TEMPLATES:
    if field in layout_by_field and field in pix_by_field:
        l,p=layout_by_field[field],pix_by_field[field]
        print(f"{field:<30} {l['support']:>4} {l['mean_anls']:>8.3f} {p['mean_anls']:>8.3f} {l['exact_match']:>7.3f} {p['exact_match']:>7.3f}")

## 22. Agreement categories

For each question classify exact-match agreement as:

- both exact;
- LayoutLM only;
- Pix2Struct only;
- neither exact.

The full row retains ANLS so near misses are not collapsed into the same interpretation as unrelated answers.

In [ ]:
layout_by_id={r["record_id"]:r for r in layout_rows}; pix_by_id={r["record_id"]:r for r in pix_rows}
agreement_rows=[]; agreement_counts=defaultdict(int)
for r in test_records:
    l,p=layout_by_id[r["id"]],pix_by_id[r["id"]]
    if l["exact_match"] and p["exact_match"]: cat="both_exact"
    elif l["exact_match"]: cat="layoutlm_only"
    elif p["exact_match"]: cat="pix2struct_only"
    else: cat="neither_exact"
    agreement_counts[cat]+=1
    agreement_rows.append({
        "record_id":r["id"],"page_id":r["page_id"],"field":r["field"],"question":r["question"],"gold_answer":r["answers"][0],
        "layoutlm_answer":l["answer"],"layoutlm_anls":l["anls"],"layoutlm_exact":l["exact_match"],
        "pix2struct_answer":p["answer"],"pix2struct_anls":p["anls"],"pix2struct_exact":p["exact_match"],
        "agreement_category":cat,
    })
print(dict(agreement_counts))

## 23. Paraphrase sensitivity

For each model, compare canonical and paraphrased ANLS plus normalized answer stability. No invariance is assumed.

In [ ]:
paraphrase_all=layout_paraphrase_rows+pix_paraphrase_rows
if paraphrase_all:
    for model_name in ("LayoutLM","Pix2Struct"):
        rows=[r for r in paraphrase_all if r["model"]==model_name]
        print({"model":model_name,"n":len(rows),
               "canonical_anls":float(np.mean([r["canonical_anls"] for r in rows])),
               "paraphrased_anls":float(np.mean([r["paraphrased_anls"] for r in rows])),
               "answer_stability":float(np.mean([r["answer_stable"] for r in rows]))})
else:
    print("Paraphrase experiment disabled.")

## 24. Unanswerable-question behavior

This probe deliberately has no gold answer and therefore no accuracy score.

Its lesson is operational: neither canonical system provides a reliable built-in abstention mechanism. A non-empty output is not evidence that the requested field exists.

In [ ]:
unanswerable_rows=layout_unanswerable_rows+pix_unanswerable_rows
for row in unanswerable_rows: print(row)

## 25. Windowing, generation and confidence semantics

LayoutLM exposes an uncalibrated `span_score` and may require multiple OCR-token windows. Pix2Struct exposes no answer confidence but does expose generated-token count and whether the configured token budget was reached.

In [ ]:
window_counts=np.array([r["n_windows"] for r in layout_rows],dtype=int)
new_tokens=np.array([r["new_tokens"] for r in pix_rows],dtype=int)
print({"layoutlm_one_window":int(np.sum(window_counts==1)),"layoutlm_multi_window":int(np.sum(window_counts>1)),
       "layoutlm_mean_windows":float(window_counts.mean()),"layoutlm_max_windows":int(window_counts.max())})
print({"pix2struct_mean_new_tokens":float(new_tokens.mean()),"pix2struct_max_new_tokens_observed":int(new_tokens.max()),
       "pix2struct_truncation_rate":float(np.mean([r["truncated"] for r in pix_rows]))})

## 26. Resource and latency comparison

Per-question latency excludes download and model-load time.

LayoutLM latency also excludes external OCR-engine time because OCR words/boxes are assumed to already exist. Pix2Struct starts from page pixels.

In [ ]:
def p95(values): return float(np.percentile(np.asarray(values,dtype=float),95))
resource_rows=[
    {"model":"LayoutLM","model_id":LAYOUTLM_ID,"revision":LAYOUTLM_REVISION,"parameter_count":layout_parameter_count,
     "weight_bytes":next(x["bytes"] for x in LAYOUTLM_MANIFEST["files"] if x["path"]=="model.safetensors"),
     "load_seconds":layout_load_seconds,"mean_latency_s":float(np.mean(layout_times)),"median_latency_s":float(np.median(layout_times)),
     "p95_latency_s":p95(layout_times),"total_eval_seconds":float(sum(layout_times)),"peak_gpu_memory_bytes":layout_peak_gpu},
    {"model":"Pix2Struct","model_id":PIX_ID,"revision":PIX_REVISION,"parameter_count":pix_parameter_count,
     "weight_bytes":next(x["bytes"] for x in PIX2STRUCT_MANIFEST["files"] if x["path"]=="model.safetensors"),
     "load_seconds":pix_load_seconds,"mean_latency_s":float(np.mean(pix_times)),"median_latency_s":float(np.median(pix_times)),
     "p95_latency_s":p95(pix_times),"total_eval_seconds":float(sum(pix_times)),"peak_gpu_memory_bytes":pix_peak_gpu},
]
for row in resource_rows: print(row)

## 27. Qualitative page panels

Select deterministic examples from the measured agreement categories. Each panel shows the receipt, light OCR boxes, LayoutLM's returned answer location, the question/gold answer, and both model outputs.

In [ ]:
def select_example(category):
    rows=[r for r in agreement_rows if r["agreement_category"]==category]
    return sorted(rows,key=lambda x:x["record_id"])[0] if rows else None
selected_examples=[x for x in [select_example("both_exact"),select_example("layoutlm_only"),select_example("pix2struct_only"),select_example("neither_exact")] if x]

def draw_qa_panel(example):
    page=test_pages[example["page_id"]]; image=page["image"].convert("RGB").copy(); draw=ImageDraw.Draw(image)
    for box in page["boxes"]: draw.rectangle(box,outline=(180,180,180),width=1)
    l=layout_by_id[example["record_id"]]
    if l["answer_union_box"] is not None: draw.rectangle(l["answer_union_box"],outline=(0,0,0),width=4)
    max_w=900
    if image.width>max_w:
        scale=max_w/image.width; image=image.resize((max_w,max(1,int(image.height*scale))))
    canvas=Image.new("RGB",(image.width,image.height+180),"white"); canvas.paste(image,(0,0)); d=ImageDraw.Draw(canvas); y=image.height+6
    lines=[f"Q: {example['question']}",f"Gold: {example['gold_answer']}",
           f"LayoutLM: {example['layoutlm_answer']}  ANLS={example['layoutlm_anls']:.3f}",
           f"Pix2Struct: {example['pix2struct_answer']}  ANLS={example['pix2struct_anls']:.3f}"]
    for line in lines: d.text((8,y),line,fill="black"); y+=38
    return canvas

examples_dir=Path(OUTPUT_DIR)/"examples"; examples_dir.mkdir(parents=True,exist_ok=True)
for ex in selected_examples:
    path=examples_dir/f"{ex['agreement_category']}_{ex['record_id']}.png"; draw_qa_panel(ex).save(path); print(path)

## 28. Release Pix2Struct before export/BYOD

The canonical comparison is now complete. Remove the model before optional user-data execution so a BYOD run can reload the two models sequentially without co-resident large checkpoints.

In [ ]:
del pix_model,pix_processor
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("Pix2Struct released.")

## 29. Machine-readable exports

The notebook writes predictions, field metrics, agreement/error categories, paraphrase and unanswerable probes, modality robustness, resource measurements, an input manifest, metrics JSON, provenance JSON, and qualitative panels.

In [ ]:
import csv
from datetime import datetime, timezone
out_dir=Path(OUTPUT_DIR); out_dir.mkdir(parents=True,exist_ok=True)

def write_csv(path,rows,fieldnames):
    with open(path,"w",encoding="utf-8",newline="") as f:
        w=csv.DictWriter(f,fieldnames=fieldnames); w.writeheader()
        for row in rows:
            out={}
            for k in fieldnames:
                value=row.get(k)
                if isinstance(value,(list,dict)): value=json.dumps(value)
                out[k]=value
            w.writerow(out)

prediction_rows=last_number_rows+keyword_rows+layout_rows+pix_rows
write_csv(out_dir/"predictions.csv",prediction_rows,[
    "record_id","page_id","field","question","gold_answer","model","answer","normalized_answer","anls","exact_match",
    "empty","answer_in_ocr","latency_seconds","start_word","end_word","span_score","n_windows","answer_union_box","new_tokens","truncated"
])
field_rows=[]
for model_name,rows in [("LayoutLM",layout_field_metrics),("Pix2Struct",pix_field_metrics)]:
    for row in rows: field_rows.append({"model":model_name,**row})
write_csv(out_dir/"field_metrics.csv",field_rows,["model","field","support","mean_anls","exact_match","empty_rate","answer_in_ocr_rate"])
write_csv(out_dir/"agreement.csv",agreement_rows,[
    "record_id","page_id","field","question","gold_answer","layoutlm_answer","layoutlm_anls","layoutlm_exact",
    "pix2struct_answer","pix2struct_anls","pix2struct_exact","agreement_category"
])
write_csv(out_dir/"paraphrase_experiment.csv",paraphrase_all,[
    "model","record_id","field","canonical_question","paraphrased_question","canonical_answer","paraphrased_answer",
    "canonical_anls","paraphrased_anls","answer_stable"
])
write_csv(out_dir/"unanswerable_probe.csv",unanswerable_rows,["model","page_id","question","answer","empty","answer_in_ocr","span_score","new_tokens","truncated"])
robustness_rows=layout_no_layout_rows+pix_degraded_rows
write_csv(out_dir/"modality_robustness.csv",robustness_rows,[
    "model","record_id","experiment","canonical_answer","variant_answer","canonical_anls","variant_anls","delta_anls","canonical_exact","variant_exact"
])
write_csv(out_dir/"resource_metrics.csv",resource_rows,[
    "model","model_id","revision","parameter_count","weight_bytes","load_seconds","mean_latency_s","median_latency_s","p95_latency_s",
    "total_eval_seconds","peak_gpu_memory_bytes"
])

input_manifest={
    "dataset":{"repo":CORD_REPO,"revision":CORD_REVISION,"test_pages":len(test_pages),"test_qa_records":len(test_records),"test_qa_digest":qa_digest},
    "observed":{
        "image_width_range":[min(p["image"].width for p in test_pages.values()),max(p["image"].width for p in test_pages.values())],
        "image_height_range":[min(p["image"].height for p in test_pages.values()),max(p["image"].height for p in test_pages.values())],
        "ocr_word_count_range":[min(len(r["words"]) for r in test_records),max(len(r["words"]) for r in test_records)],
        "question_chars_range":[min(len(r["question"]) for r in test_records),max(len(r["question"]) for r in test_records)],
    },
    "verdict":"accepted",
    "rejection_probes":[
        "LayoutLM: empty question is rejected","LayoutLM: mismatched word/box counts are rejected",
        "Pix2Struct: empty question is rejected","Pix2Struct: generation budget outside 1..128 is rejected",
    ],
}
(out_dir/"input_manifest.json").write_text(json.dumps(input_manifest,indent=2),encoding="utf-8")
metrics_export={
    "systems":system_metrics,"layoutlm_per_field":layout_field_metrics,"pix2struct_per_field":pix_field_metrics,
    "agreement_counts":dict(agreement_counts),"paraphrase":{"layoutlm":layout_paraphrase_rows,"pix2struct":pix_paraphrase_rows},
    "unanswerable":unanswerable_rows,"modality_robustness":robustness_rows,
    "layoutlm_window_summary":{"one_window":int(np.sum(window_counts==1)),"multi_window":int(np.sum(window_counts>1)),"mean":float(window_counts.mean()),"max":int(window_counts.max())},
    "pix2struct_generation_summary":{"mean_new_tokens":float(new_tokens.mean()),"max_new_tokens":int(new_tokens.max()),"truncation_rate":float(np.mean([r["truncated"] for r in pix_rows]))},
}
(out_dir/"metrics.json").write_text(json.dumps(metrics_export,indent=2),encoding="utf-8")
provenance={
    "created_utc":datetime.now(timezone.utc).isoformat(),"notebook_spec":"2.1","profile":"TASK-INFERENCE","pedagogical_mode":"WORKSHOP","comparison_scope":"MULTI-MODEL",
    "dataset":{"repo":CORD_REPO,"revision":CORD_REVISION,"license":CORD_LICENSE,"shards":CORD_FILES,"split_seed":SAMPLE_SEED,"page_split":SAMPLE_PAGE_SPLIT,
               "qa_counts":qa_counts,"test_qa_digest":qa_digest,"test_pages":[
                   {"page_id":p["id"],"source_split":p["source_split"],"source_row":p["source_row"],"image_sha256":p["pixel_sha256"],"ocr_sha256":p["ocr_sha256"]}
                   for p in page_splits["test"]]},
    "layoutlm":{"model_id":LAYOUTLM_ID,"revision":LAYOUTLM_REVISION,"weight_sha256":next(x["sha256"] for x in LAYOUTLM_MANIFEST["files"] if x["path"]=="model.safetensors"),
                "parameter_count":layout_parameter_count,"input_boundary":"question + OCR words + pixel boxes; page pixels not read"},
    "pix2struct":{"model_id":PIX_ID,"revision":PIX_REVISION,"weight_sha256":next(x["sha256"] for x in PIX2STRUCT_MANIFEST["files"] if x["path"]=="model.safetensors"),
                  "parameter_count":pix_parameter_count,"header_font_sha256":FONT_SHA256,"max_new_tokens":MAX_NEW_TOKENS,
                  "input_boundary":"page pixels + question rendered as header; no external OCR input"},
    "question_templates":QUESTION_TEMPLATES,"anls_threshold":ANLS_THRESHOLD,"runtime":RUNTIME,"resource_metrics":resource_rows,
}
(out_dir/"provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")
print("Exports:")
for p in sorted(out_dir.iterdir()): print(" ",p)

## 30. Bring Your Own Document QA data (optional)

The common comparison requires **both** page pixels and OCR/layout data.

Expected structure:

```text
dataset/
  pages/
    page001.png
  records.jsonl
```

Each JSONL record must contain `id`, `page_id`, `file`, `question`, `words`, and `boxes`; `answers` is optional. Without accepted answers the two models still run but evaluation is `not-measurable`.

The notebook intentionally does **not** install or run Tesseract. LayoutLM's OCR remains caller-owned.

### Privacy

Document pages may contain names, addresses, financial data, account identifiers, medical information, signatures, or proprietary records. Do not upload confidential, restricted, personal, or regulated documents to a hosted notebook runtime unless authorized.

In [ ]:
byod_result=None
if USE_BYOD:
    root=Path(BYOD_PATH); records_path=root/"records.jsonl"; pages_dir=root/"pages"
    if not records_path.is_file() or not pages_dir.is_dir():
        raise FileNotFoundError("BYOD_PATH must contain pages/ and records.jsonl")
    raw_records=[json.loads(line) for line in records_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not 1<=len(raw_records)<=500: raise ValueError("BYOD accepts 1..500 QA records")
    byod_records=[]
    for i,r in enumerate(raw_records):
        required={"id","page_id","file","question","words","boxes"}
        if not required.issubset(r): raise ValueError(f"BYOD record {i} missing {sorted(required-set(r))}")
        image=Image.open(pages_dir/Path(r["file"]).name); image.load(); image=image.convert("RGB")
        if not r["words"] or len(r["words"])!=len(r["boxes"]): raise ValueError(f"BYOD record {i}: words/boxes mismatch")
        if not 1<=len(r["words"])<=2000: raise ValueError(f"BYOD record {i}: OCR word count outside 1..2000")
        for box in r["boxes"]:
            x0,y0,x1,y1=[float(v) for v in box]
            if not (0<=x0<=x1<=image.width and 0<=y0<=y1<=image.height): raise ValueError(f"BYOD record {i}: box outside image")
        byod_records.append({**r,"image":image,"image_size":[image.width,image.height],"answers":r.get("answers")})

    lt=AutoTokenizer.from_pretrained(str(LAYOUTLM_DIR),local_files_only=True,trust_remote_code=False)
    lm=LayoutLMForQuestionAnswering.from_pretrained(str(LAYOUTLM_DIR),local_files_only=True,trust_remote_code=False,dtype=torch.float32).to(DEVICE).eval()
    layout_tokenizer,layout_model,layout_sep_id=lt,lm,lt.sep_token_id
    layout_byod=[]
    for r in byod_records:
        result=layout_answer(r["question"],r["words"],r["boxes"],r["image_size"]); layout_byod.append({"id":r["id"],"answer":result["answer"]})
    del layout_model,layout_tokenizer; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    pp=Pix2StructProcessor.from_pretrained(str(PIX2STRUCT_DIR),local_files_only=True,trust_remote_code=False)
    pm=Pix2StructForConditionalGeneration.from_pretrained(str(PIX2STRUCT_DIR),local_files_only=True,trust_remote_code=False,dtype=torch.float32).to(DEVICE).eval()
    pix_processor,pix_model=pp,pm
    pix_byod=[]
    for r in byod_records:
        result=pix_answer(r["image"],r["question"]); pix_byod.append({"id":r["id"],"answer":result["answer"]})
    labelled=all(isinstance(r.get("answers"),list) and r["answers"] for r in byod_records)
    byod_result={"n":len(byod_records),"evaluation_verdict":"measured" if labelled else "not-measurable","layoutlm":layout_byod,"pix2struct":pix_byod}
    if labelled:
        byod_result["layoutlm_anls"]=float(np.mean([anls(x["answer"],r["answers"]) for x,r in zip(layout_byod,byod_records,strict=True)]))
        byod_result["pix2struct_anls"]=float(np.mean([anls(x["answer"],r["answers"]) for x,r in zip(pix_byod,byod_records,strict=True)]))
    print({k:v for k,v in byod_result.items() if k not in {"layoutlm","pix2struct"}})
else:
    print("BYOD disabled on the canonical Run all path.")

## 31. Interpretation and limitations

### OCR-free is not dependency-free

Pix2Struct removes external OCR but depends directly on page rendering quality, patch resolution and generation behavior.

### LayoutLM offers stronger traceability

Its answer maps to exact OCR word indices and boxes. That can matter operationally even when text-level metrics are similar.

### Generated answers can leave the page

Pix2Struct may produce text that the supplied OCR does not contain. This may reflect normalization, OCR disagreement, or hallucination. The canonical API exposes no answer-confidence signal.

### The evaluation structurally favors extractive answers

CORD questions are created only when the gold answer is a contiguous OCR value span. This must accompany every comparative interpretation.

### Neither system reliably abstains

The unanswerable probe is intentionally not scored. Both systems can return plausible-looking answers even when the requested field is absent.

### ANLS is not factual-safety evidence

A high average ANLS does not justify automatic consequential use of extracted values without domain validation and appropriate review.

## 32. Terminal summary

The final cell verifies required outputs and prints measurements from the current execution without choosing a winner.

In [ ]:
required_outputs=[
    out_dir/"predictions.csv",out_dir/"field_metrics.csv",out_dir/"agreement.csv",out_dir/"paraphrase_experiment.csv",
    out_dir/"unanswerable_probe.csv",out_dir/"modality_robustness.csv",out_dir/"resource_metrics.csv",
    out_dir/"input_manifest.json",out_dir/"metrics.json",out_dir/"provenance.json",
]
missing=[str(p) for p in required_outputs if not p.is_file()]
if missing: raise RuntimeError(f"Required outputs missing: {missing}")
print("DIMER Document Question Answering Workshop")
print("-"*45)
print(f"Test pages: {len(test_pages)}")
print(f"QA records: {len(test_records)}")
print(f"Fields represented: {len({r['field'] for r in test_records})}")
print()
print(f"{'System':<24} {'ANLS':>8} {'Exact':>8} {'Empty':>8}")
print("-"*52)
for name in ("Last-number baseline","Keyword lookup","LayoutLM","Pix2Struct"):
    m=system_metrics[name]; print(f"{name:<24} {m['anls']:>8.3f} {m['exact_match']:>8.3f} {m['empty_rate']:>8.3f}")
print()
print("LayoutLM")
print("  OCR required: yes")
print("  answer localization: yes")
print(f"  mean latency: {np.mean(layout_times):.4f} s")
print("Pix2Struct")
print("  OCR required: no")
print("  answer localization: no")
print(f"  mean latency: {np.mean(pix_times):.4f} s")
print(f"  truncation rate: {np.mean([r['truncated'] for r in pix_rows]):.4f}")
print(f"Outputs: {out_dir}/")

## Try it yourself — one controlled change

Use the same experimental discipline as the canonical path:

**Predict → change one variable → rerun → observe → explain**

Use the existing LayoutLM modality experiment and remove spatial layout while keeping the OCR words and question unchanged. Predict which question types should be most affected, rerun that comparison, then explain what the result says about the value—and limits—of spatial information.

Keep exploratory changes separate from the frozen canonical test result.


## Self-paced checkpoint

Before opening the sample interpretation, answer:

1. What did the model/system receive as input, and what did it produce?
2. Which baseline/reference tells you whether the learned model added value?
3. What failure mode or tradeoff matters most here?
4. What additional evidence would you want before transferring the result to a new domain?

<details>
<summary><b>Show a sample interpretation</b></summary>

The two systems use different evidence and answer mechanisms. LayoutLM depends on OCR words plus layout; Pix2Struct generates from page pixels and the question. Generated answers can leave the page, neither model reliably abstains, and ANLS similarity is not evidence of factual safety.

Use the outputs from **your run** when writing your final answer; small numeric differences across supported runtimes are possible.

</details>


## Write an evidence-based conclusion

1. **State the question** tested by this notebook.
2. **Report the primary result** against the relevant baseline/reference.
3. **Add supporting evidence** from a secondary metric, error pattern, disagreement, or qualitative diagnostic.
4. **Account for cost/complexity** when it materially affects the comparison.
5. **State the limits** of the data, split, model revision, and configuration.

Compare the held-out answer results using the common metrics, identify at least one field/question type where the systems differ, describe the modality experiment, and state the traceability/abstention limitations before suggesting deployment.


# Troubleshooting

| What you see | Likely cause | What to do |
|---|---|---|
| Accelerator unavailable or execution is unexpectedly slow | The runtime does not match the documented resource envelope | Select the documented accelerator/runtime, start a fresh session, and run top-to-bottom. |
| Package/version or stale-module error | Incompatible libraries were already imported in the hosted kernel | Start a fresh runtime and choose **Run all** before importing extra packages. Do not bypass version checks. |
| Model/sample digest or byte-size check fails | Download is incomplete or upstream bytes differ from the pinned artifact | Remove the affected runtime cache/download and rerun. Do not disable the integrity check. |
| Out-of-memory or runtime restart | Too many large models/intermediates are resident | Use the default tier, follow explicit unload/release steps, and avoid combining optional heavy branches. |
| BYOD validation fails | Input does not satisfy the documented schema, shape, labels, or limits | Follow the validation message, correct the indicated field/format, then rerun the BYOD branch. |
| Your numbers differ slightly | Supported hardware/library execution can introduce small numerical variation | Verify the split, model revision, metric definition, and qualitative pattern before treating the difference as substantive. |


# Glossary

| Term | Meaning in this notebook |
|---|---|
| **Document QA** | Answering a natural-language question using information in a document page. |
| **Extractive QA** | Selecting an answer span from supplied text evidence. |
| **Generative QA** | Producing answer tokens rather than selecting a fixed text span. |
| **Layout** | Spatial positions of words or regions on the document page. |
| **ANLS** | Average Normalized Levenshtein Similarity; a string-similarity metric commonly used for document QA. |
| **Abstention** | Choosing not to answer when evidence is missing or uncertain. |